# Logical Error Rate Comparison: conv_max_conf

Compare logical error rates for the bivariate bicycle code.

- **conv_max_conf=True**: Use `pred_bp` if BP converged, else use `pred`
- **conv_max_conf=False**: Always use `pred` (LSD output)
- **Original ldpc**: Use the original `BpLsdDecoder` from the `ldpc` package directly

In [7]:
import numpy as np
from tqdm import tqdm

from ldpc.bplsd_decoder import BpLsdDecoder
from ldpc_post_selection.bplsd_decoder import SoftOutputsBpLsdDecoder
from simulations.utils.build_circuit import build_BB_circuit, get_BB_distance

## Setup

In [8]:
# Parameters
n = 72
p = 0.001
T = get_BB_distance(n)  # T = 12 (code distance)
shots = 1_000_000

print(f"Code: [[{n}, 12, {T}]]")
print(f"Physical error rate: p = {p}")
print(f"Number of shots: {shots:,}")

Code: [[72, 12, 6]]
Physical error rate: p = 0.001
Number of shots: 1,000,000


In [9]:
# Build circuit
circuit = build_BB_circuit(n=n, T=T, p=p)
print(f"Number of detectors: {circuit.num_detectors}")
print(f"Number of observables: {circuit.num_observables}")

Number of detectors: 252
Number of observables: 12


In [10]:
# Initialize decoder
decoder_params = {
    "max_iter": 30,
    "bp_method": "minimum_sum",
    "lsd_method": "LSD_0",
    "lsd_order": 0,
}

decoder = SoftOutputsBpLsdDecoder(circuit=circuit, merge_duplicate_errors=True, **decoder_params)
obs_matrix_T = decoder.obs_matrix.T

print(f"Decoder initialized with params: {decoder_params}")

Decoder initialized with params: {'max_iter': 30, 'bp_method': 'minimum_sum', 'lsd_method': 'LSD_0', 'lsd_order': 0}


## Sample and Decode

In [11]:
# Sample detector and observable outcomes
sampler = circuit.compile_detector_sampler()
det, obs = sampler.sample(shots, separate_observables=True)

print(f"Detector outcomes shape: {det.shape}")
print(f"Observable outcomes shape: {obs.shape}")

Detector outcomes shape: (1000000, 252)
Observable outcomes shape: (1000000, 12)


In [12]:
# Decode all samples
preds = []
preds_bp = []
converges = []

for det_single in tqdm(det, desc="Decoding"):
    pred, pred_bp, converge, _ = decoder.decode(
        det_single,
        include_cluster_stats=False,
        compute_logical_gap_proxy=False,
    )
    preds.append(pred)
    preds_bp.append(pred_bp)
    converges.append(converge)

preds = np.array(preds)
preds_bp = np.array(preds_bp)
converges = np.array(converges)

print(f"\nConvergence rate: {converges.mean():.4f}")

Decoding: 100%|██████████| 1000000/1000000 [05:44<00:00, 2903.84it/s]



Convergence rate: 0.9929


## Compute Logical Error Rates

In [13]:
def compute_logical_errors(
    predictions: np.ndarray, obs_actual: np.ndarray, obs_matrix_T: np.ndarray
) -> np.ndarray:
    """
    Compute logical errors for given predictions.

    Parameters
    ----------
    predictions : 2D numpy array of bool with shape (shots, num_errors)
        Predicted error patterns.
    obs_actual : 2D numpy array of bool with shape (shots, num_observables)
        Actual observable outcomes.
    obs_matrix_T : 2D numpy array with shape (num_errors, num_observables)
        Transposed observable matrix.

    Returns
    -------
    logical_errors : 1D numpy array of bool with shape (shots,)
        True if any observable was incorrectly predicted.
    """
    obs_predicted = ((predictions.astype(np.uint8) @ obs_matrix_T) % 2).astype(bool)
    logical_errors = np.any(obs_actual ^ obs_predicted, axis=1)
    return logical_errors

In [14]:
# conv_max_conf=False: Always use pred (LSD output)
logical_errors_lsd = compute_logical_errors(preds, obs, obs_matrix_T)
ler_lsd = logical_errors_lsd.mean()
ler_lsd_std = np.sqrt(ler_lsd * (1 - ler_lsd) / shots)

print(f"conv_max_conf=False (always LSD):")
print(f"  Logical error rate: {ler_lsd:.6f} ± {ler_lsd_std:.6f}")
print(f"  Logical errors: {logical_errors_lsd.sum():,} / {shots:,}")

conv_max_conf=False (always LSD):
  Logical error rate: 0.000243 ± 0.000016
  Logical errors: 243 / 1,000,000


In [15]:
# conv_max_conf=True: Use pred_bp if converged, else pred
preds_conv_max_conf = np.where(converges[:, None], preds_bp, preds)
logical_errors_conv = compute_logical_errors(preds_conv_max_conf, obs, obs_matrix_T)
ler_conv = logical_errors_conv.mean()
ler_conv_std = np.sqrt(ler_conv * (1 - ler_conv) / shots)

print(f"conv_max_conf=True (BP if converged, else LSD):")
print(f"  Logical error rate: {ler_conv:.6f} ± {ler_conv_std:.6f}")
print(f"  Logical errors: {logical_errors_conv.sum():,} / {shots:,}")

conv_max_conf=True (BP if converged, else LSD):
  Logical error rate: 0.000246 ± 0.000016
  Logical errors: 246 / 1,000,000


## Original ldpc Package Decoder

In [16]:
# Initialize original ldpc decoder
original_decoder = BpLsdDecoder(
    decoder.H,
    error_channel=decoder.priors,
    max_iter=decoder_params["max_iter"],
    bp_method=decoder_params["bp_method"],
    lsd_method=decoder_params["lsd_method"],
    lsd_order=decoder_params["lsd_order"],
)

# Decode with original ldpc decoder
preds_original = []
for det_single in tqdm(det, desc="Decoding (original ldpc)"):
    pred_orig = original_decoder.decode(det_single)
    preds_original.append(pred_orig)

preds_original = np.array(preds_original, dtype=bool)

print(f"Original ldpc decoder predictions shape: {preds_original.shape}")

Decoding (original ldpc): 100%|██████████| 1000000/1000000 [03:30<00:00, 4740.76it/s]


Original ldpc decoder predictions shape: (1000000, 2232)


In [17]:
# Original ldpc: Compute logical error rate
logical_errors_original = compute_logical_errors(preds_original, obs, obs_matrix_T)
ler_original = logical_errors_original.mean()
ler_original_std = np.sqrt(ler_original * (1 - ler_original) / shots)

print(f"Original ldpc (BpLsdDecoder):")
print(f"  Logical error rate: {ler_original:.6f} ± {ler_original_std:.6f}")
print(f"  Logical errors: {logical_errors_original.sum():,} / {shots:,}")

Original ldpc (BpLsdDecoder):
  Logical error rate: 0.000246 ± 0.000016
  Logical errors: 246 / 1,000,000
